# Baseline Modeling FD001

Experiment notebook for the existing Ridge baseline on NASA C-MAPSS FD001. Model construction, RUL labeling, splitting, metrics, and official-test alignment all use package code; this notebook handles orchestration and visualization.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from turbofan.config.schema import load_config
from turbofan.data.loader import load_raw_test, load_raw_train, load_rul_labels
from turbofan.models.baseline import build_baseline_pipeline
from turbofan.models.evaluate import (
    add_rul_column,
    align_official_test_labels,
    select_last_cycle_per_engine,
    split_features_target,
)
from turbofan.models.metrics import regression_metrics
from turbofan.models.split import split_by_engine

pd.options.display.max_columns = 120
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

print(f"Project root: {PROJECT_ROOT}")

## 2. Load Configuration and Training Data

In [ ]:
cfg = load_config(Path("configs/default.yaml"))
train_raw = load_raw_train(cfg.data)

print(f"Subset: {cfg.data.fd_subset}")
print(f"Raw train rows: {len(train_raw):,}")
print(f"Train engines: {train_raw['engine_id'].nunique():,}")
print(f"Sensor std threshold: {cfg.features.sensor_std_threshold}")
print(f"Sensor keep list: {cfg.features.sensor_keep}")
train_raw.head()

## 3. Add RUL Labels and Split by Engine

In [ ]:
train_labeled = add_rul_column(train_raw, max_rul=cfg.data.max_rul)
train_df, val_df = split_by_engine(
    train_labeled,
    test_size=cfg.data.test_size,
    random_seed=cfg.data.random_seed,
)

print(f"Train split rows: {len(train_df):,}")
print(f"Validation split rows: {len(val_df):,}")
print(f"Train engines: {train_df['engine_id'].nunique():,}")
print(f"Validation engines: {val_df['engine_id'].nunique():,}")

In [ ]:
pd.Series(train_df.columns, name="column").to_frame().head(12)

## 4. Inspect Low-Variance Sensors and Train Ridge Baseline

In [ ]:
sensor_cols = [col for col in train_df.columns if col.startswith("s_")]
sensor_std = train_df[sensor_cols].std().sort_values()
low_variance_sensors = sensor_std[
    sensor_std <= cfg.features.sensor_std_threshold
]

print(f"Sensor columns: {len(sensor_cols)}")
print(
    "Low-variance sensors at threshold "
    f"{cfg.features.sensor_std_threshold}: {len(low_variance_sensors)}"
)

sensor_std_frame = sensor_std.rename("std").to_frame()
sensor_std_frame["will_drop"] = sensor_std_frame.index.isin(
    low_variance_sensors.index.difference(cfg.features.sensor_keep)
)
sensor_std_frame.head(10)

In [ ]:
X_train, y_train = split_features_target(train_df)
X_val, y_val = split_features_target(val_df)

estimator = build_baseline_pipeline(
    model_name=cfg.model.name,
    alpha=cfg.model.alpha,
    sensor_std_threshold=cfg.features.sensor_std_threshold,
    sensor_keep=cfg.features.sensor_keep,
)
estimator.fit(X_train, y_train)

sensor_dropper = estimator.named_steps["features"].named_steps["sensor_dropper"]
print(f"Dropped sensors: {sensor_dropper.columns_to_drop_}")

val_pred = np.clip(
    np.asarray(estimator.predict(X_val), dtype=np.float64),
    0.0,
    None,
)
val_metrics = regression_metrics(y_val, val_pred)
pd.Series(val_metrics, name="validation")

## 5. Validation Prediction Frame

In [ ]:
val_predictions = X_val[["engine_id", "cycle"]].copy()
val_predictions["rul"] = y_val.to_numpy(dtype=np.float64)
val_predictions["prediction"] = val_pred
val_predictions["residual"] = (
    val_predictions["prediction"] - val_predictions["rul"]
)

val_predictions.head()

## 6. Actual vs Predicted RUL

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(
    val_predictions["rul"],
    val_predictions["prediction"],
    alpha=0.35,
    s=14,
)
limit = float(
    max(val_predictions["rul"].max(), val_predictions["prediction"].max())
)
ax.plot([0.0, limit], [0.0, limit], color="black", linestyle="--", linewidth=1)
ax.set_xlim(0.0, limit)
ax.set_ylim(0.0, limit)
ax.set_xlabel("Actual RUL")
ax.set_ylabel("Predicted RUL")
ax.set_title("Validation Actual vs Predicted RUL")
plt.show()

## 7. Residual Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(val_predictions["residual"], bins=40, color="#4c78a8")
axes[0].axvline(0.0, color="black", linestyle="--", linewidth=1)
axes[0].set_xlabel("Prediction - actual RUL")
axes[0].set_ylabel("Rows")
axes[0].set_title("Residual Distribution")

axes[1].scatter(
    val_predictions["rul"],
    val_predictions["residual"],
    alpha=0.35,
    s=14,
)
axes[1].axhline(0.0, color="black", linestyle="--", linewidth=1)
axes[1].set_xlabel("Actual RUL")
axes[1].set_ylabel("Prediction - actual RUL")
axes[1].set_title("Residuals vs True RUL")

fig.tight_layout()
plt.show()

## 8. Per-Engine Validation Trajectories

In [ ]:
trajectory_engine_ids = (
    val_predictions["engine_id"].drop_duplicates().sort_values().head(4).to_list()
)

fig, axes = plt.subplots(
    len(trajectory_engine_ids),
    1,
    figsize=(10, 2.4 * len(trajectory_engine_ids)),
    sharex=False,
)
if len(trajectory_engine_ids) == 1:
    axes = [axes]

for ax, engine_id in zip(axes, trajectory_engine_ids, strict=True):
    engine_rows = val_predictions.loc[
        val_predictions["engine_id"] == engine_id
    ].sort_values("cycle")
    ax.plot(engine_rows["cycle"], engine_rows["rul"], label="actual")
    ax.plot(engine_rows["cycle"], engine_rows["prediction"], label="predicted")
    ax.set_title(f"Engine {engine_id}")
    ax.set_xlabel("Cycle")
    ax.set_ylabel("RUL")
    ax.legend(loc="best")

fig.tight_layout()
plt.show()

## 9. Optional Official Test Evaluation

This cell evaluates final-cycle predictions when `test_FD001.txt` and `RUL_FD001.txt` exist in the configured raw data directory. It follows the same helper-based flow as the training script.

In [ ]:
official_test_result = None

try:
    test_raw = load_raw_test(cfg.data)
    rul_labels = load_rul_labels(cfg.data)
except FileNotFoundError as exc:
    print(f"Official test evaluation skipped: {exc}")
else:
    test_last_rows = select_last_cycle_per_engine(test_raw)
    official_y = align_official_test_labels(test_last_rows, rul_labels)

    test_pred_all = np.clip(
        np.asarray(estimator.predict(test_raw), dtype=np.float64),
        0.0,
        None,
    )
    test_pred_rows = test_raw[["engine_id", "cycle"]].copy()
    test_pred_rows["prediction"] = test_pred_all
    test_last_pred_rows = select_last_cycle_per_engine(test_pred_rows)
    official_pred = np.clip(
        test_last_pred_rows["prediction"].to_numpy(dtype=np.float64),
        0.0,
        None,
    )

    official_metrics = regression_metrics(official_y, official_pred)
    official_predictions = test_last_rows[["engine_id", "cycle"]].copy()
    official_predictions["rul"] = official_y.to_numpy(dtype=np.float64)
    official_predictions["prediction"] = official_pred
    official_predictions["residual"] = (
        official_predictions["prediction"] - official_predictions["rul"]
    )
    official_test_result = {
        "metrics": official_metrics,
        "predictions": official_predictions,
    }

    display(pd.Series(official_metrics, name="official_test"))
    display(official_predictions.head())